In [ ]:
"""
data_generation.py
==================
Generates a realistic, unbiased synthetic personal finance dataset
for the AI-Driven Financial Risk & Intelligence System.

Structure:
- 5 user segments (personas)
- 2000 users total
- 24 months history per user
- Output: data/raw/finance_data.csv
"""

import numpy as np
import pandas as pd
import os

# ─────────────────────────────────────────────
# SEED for reproducibility
# ─────────────────────────────────────────────
np.random.seed(42)

# ─────────────────────────────────────────────
# STEP 1: DEFINE USER SEGMENTS (PERSONAS)
# Each segment has its own statistical behavior
# ─────────────────────────────────────────────

SEGMENTS = {

    "stable_salaried": {
        "description": "Stable salaried professionals. Consistent income, moderate expenses, decent savers.",
        "count": 500,
        "income_mean": 75000,
        "income_std": 10000,
        "income_log_scale": 11.2,     # exp(11.2) ≈ ₹73k base — clamped to 500k max
        "rent_pct": (0.25, 0.05),
        "food_pct": (0.15, 0.03),
        "travel_pct": (0.08, 0.02),
        "utilities_pct": (0.06, 0.01),
        "other_pct": (0.07, 0.02),
        "income_growth": 0.003,
        "volatility_level": "low",
        # ── Age: mid-career professionals ──
        "age_mean": 34,
        "age_std": 7,
        "age_min": 26,
        "age_max": 55,
    },

    "gig_volatile": {
        "description": "Gig workers / freelancers. Highly volatile income, inconsistent expenses.",
        "count": 400,
        "income_log_scale": 10.8,
        "income_std": 18000,
        "rent_pct": (0.28, 0.07),
        "food_pct": (0.18, 0.06),
        "travel_pct": (0.12, 0.05),
        "utilities_pct": (0.07, 0.03),
        "other_pct": (0.10, 0.05),
        "income_growth": 0.001,
        "volatility_level": "high",
        # ── Age: younger, early career ──
        "age_mean": 27,
        "age_std": 5,
        "age_min": 22,
        "age_max": 40,
    },

    "high_earner_lifestyle": {
        "description": "High earners with lifestyle inflation. Big income, bigger expenses.",
        "count": 350,
        "income_log_scale": 11.8,
        "income_std": 20000,
        "rent_pct": (0.30, 0.05),
        "food_pct": (0.20, 0.04),
        "travel_pct": (0.18, 0.06),
        "utilities_pct": (0.08, 0.02),
        "other_pct": (0.15, 0.05),
        "income_growth": 0.005,
        "volatility_level": "medium",
        # ── Age: established professionals / senior roles ──
        "age_mean": 40,
        "age_std": 7,
        "age_min": 30,
        "age_max": 58,
    },

    "conservative_saver": {
        "description": "Disciplined savers. Low to mid income, very controlled expenses.",
        "count": 400,
        "income_log_scale": 10.9,
        "income_std": 8000,
        "rent_pct": (0.20, 0.03),
        "food_pct": (0.12, 0.02),
        "travel_pct": (0.05, 0.02),
        "utilities_pct": (0.05, 0.01),
        "other_pct": (0.05, 0.02),
        "income_growth": 0.002,
        "volatility_level": "low",
        # ── Age: wide range — discipline is age-independent ──
        "age_mean": 38,
        "age_std": 10,
        "age_min": 24,
        "age_max": 60,
    },

    "financially_struggling": {
        "description": "Financially stressed users. Low income, high expense ratio, minimal savings.",
        "count": 350,
        "income_log_scale": 10.5,
        "income_std": 7000,
        "rent_pct": (0.38, 0.06),
        "food_pct": (0.25, 0.05),
        "travel_pct": (0.10, 0.04),
        "utilities_pct": (0.09, 0.03),
        "other_pct": (0.12, 0.04),
        "income_growth": 0.0005,
        "volatility_level": "high",
        # ── Age: skewed younger but spans full range ──
        "age_mean": 31,
        "age_std": 9,
        "age_min": 22,
        "age_max": 55,
    },
}

MONTHS = 24   # History per user


# ─────────────────────────────────────────────
# STEP 2: HELPER — CLAMP VALUES
# Prevent negative income/expense values
# ─────────────────────────────────────────────

def clamp(value, min_val, max_val):
    return max(min_val, min(value, max_val))


# ─────────────────────────────────────────────
# STEP 3: VOLATILITY NOISE MULTIPLIER
# Controls month-to-month randomness per segment
# ─────────────────────────────────────────────

VOLATILITY_NOISE = {
    "low": 0.03,
    "medium": 0.07,
    "high": 0.15,
}


# ─────────────────────────────────────────────
# STEP 4: SIMULATE ONE USER
# Generates 24 months of financial records for a single user
# ─────────────────────────────────────────────

def simulate_user(user_id, segment_name, segment, age, city_tier):
    """
    Simulate monthly financial records for one user.

    Returns:
        List of dicts, one per month.
    """

    noise_level = VOLATILITY_NOISE[segment["volatility_level"]]

    # Base income from log-normal distribution (realistic salary skew)
    base_income = np.random.lognormal(mean=segment["income_log_scale"], sigma=0.3)
    base_income = clamp(base_income, 15000, 500000)

    # City tier adjustments — metro is more expensive
    city_multiplier = {"metro": 1.20, "tier2": 1.00, "tier3": 0.82}[city_tier]

    records = []

    for month in range(1, MONTHS + 1):

        # Gradual income growth with random noise
        income_growth_factor = 1 + (segment["income_growth"] * month)
        monthly_noise = np.random.normal(0, noise_level)
        income = base_income * income_growth_factor * (1 + monthly_noise)
        income = clamp(income, 10000, 600000)

        # Occasionally simulate income shocks (job loss, bonus, contract end)
        if np.random.rand() < 0.03:   # 3% chance any month
            shock_type = np.random.choice(["drop", "spike"])
            income = income * (0.5 if shock_type == "drop" else 1.4)
            income = clamp(income, 8000, 600000)

        # Generate expense categories
        def expense(pct_params):
            mean_pct, std_pct = pct_params
            pct = np.random.normal(mean_pct, std_pct)
            pct = clamp(pct, 0.01, 0.80)  # cap any single category
            return round(income * pct * city_multiplier, 2)

        rent       = expense(segment["rent_pct"])
        food       = expense(segment["food_pct"])
        travel     = expense(segment["travel_pct"])
        utilities  = expense(segment["utilities_pct"])
        other      = expense(segment["other_pct"])

        # Occasionally add a big irregular expense (medical, wedding, etc.)
        irregular_expense = 0
        if np.random.rand() < 0.05:   # 5% chance any month
            irregular_expense = round(np.random.uniform(2000, 30000), 2)

        total_expense = round(rent + food + travel + utilities + other + irregular_expense, 2)
        savings       = round(income - total_expense, 2)

        records.append({
            "user_id":           user_id,
            "segment":           segment_name,
            "age":               age,
            "city_tier":         city_tier,
            "month":             month,
            "income":            round(income, 2),
            "rent":              rent,
            "food":              food,
            "travel":            travel,
            "utilities":         utilities,
            "other_expense":     other,
            "irregular_expense": irregular_expense,
            "total_expense":     total_expense,
            "savings":           savings,
        })

    return records


# ─────────────────────────────────────────────
# STEP 5: GENERATE FULL DATASET
# Loop through all segments and generate users
# ─────────────────────────────────────────────

def generate_dataset():
    all_records = []
    user_id = 1

    city_tiers   = ["metro", "tier2", "tier3"]
    city_weights = [0.40, 0.35, 0.25]   # more users in metros

    for segment_name, segment in SEGMENTS.items():
        print(f"  Generating segment: {segment_name} ({segment['count']} users)...")

        for _ in range(segment["count"]):
            age = int(np.random.normal(segment["age_mean"], segment["age_std"]))
            age = clamp(age, segment["age_min"], segment["age_max"])
            city_tier = np.random.choice(city_tiers, p=city_weights)

            records = simulate_user(user_id, segment_name, segment, age, city_tier)
            all_records.extend(records)
            user_id += 1

    df = pd.DataFrame(all_records)

    # Reorder columns cleanly
    col_order = [
        "user_id", "segment", "age", "city_tier", "month",
        "income", "rent", "food", "travel", "utilities",
        "other_expense", "irregular_expense", "total_expense", "savings"
    ]
    df = df[col_order]

    return df


# ─────────────────────────────────────────────
# STEP 6: SAVE + BASIC VALIDATION
# ─────────────────────────────────────────────

def validate_dataset(df):
    print("\n── DATASET VALIDATION ─────────────────────")
    print(f"  Total rows       : {len(df):,}")
    print(f"  Unique users     : {df['user_id'].nunique():,}")
    print(f"  Months per user  : {df.groupby('user_id')['month'].count().unique().tolist()}")
    print(f"  Negative savings : {(df['savings'] < 0).sum():,} rows ({(df['savings'] < 0).mean()*100:.1f}%)")
    print(f"  Segments         :\n{df.groupby('segment')['user_id'].nunique().to_string()}")
    print(f"\n  Income stats (₹):")
    print(df.groupby('segment')['income'].mean().apply(lambda x: f"  ₹{x:,.0f}").to_string())
    print("\n  Expense ratio by segment:")
    df['expense_ratio'] = df['total_expense'] / df['income']
    print(df.groupby('segment')['expense_ratio'].mean().apply(lambda x: f"  {x:.2%}").to_string())

    print("\n  Age distribution by segment:")
    age_df = df.drop_duplicates('user_id')
    print(age_df.groupby('segment')['age'].agg(['mean','min','max'])
          .apply(lambda r: f"  mean={r['mean']:.1f}, range=[{r['min']}–{r['max']}]", axis=1)
          .to_string())
    print("────────────────────────────────────────────\n")


if __name__ == "__main__":
    print("\n🔄 Generating synthetic financial dataset...\n")

    os.makedirs("data/raw", exist_ok=True)

    df = generate_dataset()

    validate_dataset(df)

    # ── LEAKAGE PROTECTION ──────────────────────────────────────────────────
    # Save TWO versions:
    #
    # 1. finance_data_full.csv  → includes 'segment' column
    #    Use ONLY for: validation, auditing, viva explanation
    #    NEVER feed into ML model
    #
    # 2. finance_data.csv       → segment column REMOVED
    #    Use for: all ML training, feature engineering, forecasting
    #    This is the clean, leakage-free dataset
    # ────────────────────────────────────────────────────────────────────────

    full_path  = "data/raw/finance_data_full.csv"   # with segment (audit only)
    clean_path = "data/raw/finance_data.csv"         # without segment (ML safe)

    df.to_csv(full_path, index=False)
    df.drop(columns=["segment"]).to_csv(clean_path, index=False)

    print(f"✅ Full dataset saved  → {full_path}  (DO NOT use for ML)")
    print(f"✅ Clean dataset saved → {clean_path}  (use this for ML)")
    print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
    print("⚠️  IMPORTANT: Always use finance_data.csv for training.")
    print("   finance_data_full.csv is for validation and viva only.\n")